### **Multimodal Agentic RAG**
**Stack:** Unstructured.io · NVIDIA bge-m3 (dense 1024d) · FastEmbed BM25 (sparse) · Qdrant · ChatNVIDIA


In [1]:
# # Run once — install all required packages
# %pip install -q \
#     qdrant-client \
#     "langchain-qdrant>=0.2.0" \
#     langchain-nvidia-ai-endpoints \
#     fastembed \
#     "unstructured[pdf]" \
#     unstructured-pytesseract \
#     langchain-core \
#     langchain-community \
#     python-dotenv

# print("All dependencies installed.")

In [18]:
from dotenv import load_dotenv
import os

load_dotenv()

# Required keys in your .env:
#   NVIDIA_API_KEY=...
#   QDRANT_CLUSTER_URL=https://xxxx.qdrant.io
#   QDRANT_API_KEY=...

QDRANT_URL     = os.getenv("QDRANT_CLUSTER_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")


In [4]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")
result = llm.invoke("Plan a three-step research workflow for competitive analysis.")
print(result.content)

Here's a three-step research workflow for competitive analysis:

**Step 1: Gather Information and Identify Competitors (Research and Data Collection)**

1. **Define the scope**: Identify the specific market or industry you're analyzing and the key competitors.
2. **Conduct a competitor list**: Research and compile a list of direct and indirect competitors, including their company names, websites, and social media profiles.
3. **Gather publicly available data**: Collect information from publicly available sources, such as:
	* Company websites and social media profiles
	* Online reviews and ratings
	* Industry reports and news articles
	* Social media listening tools
4. **Identify key performance indicators (KPIs)**: Determine the most relevant KPIs to track for your competitive analysis, such as:
	* Revenue growth
	* Customer acquisition costs
	* Customer retention rates
	* Market share

**Step 2: Analyze Competitor Strengths and Weaknesses (Competitor Profiling)**

1. **Create a compet

Test Dense Embeddings (NVIDIA baai/bge-m3 → 1024-dim)

In [5]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

# baai/bge-m3 → 1024-dimensional dense vectors
embeddings = NVIDIAEmbeddings(model="baai/bge-m3")

vector = embeddings.embed_query("What is competitive analysis?")
print(f"Vector dimensions : {len(vector)}")
print(f"First 10 values   : {vector[:10]}")
print(f"Last  10 values   : {vector[-10:]}")
print(f"Min value         : {min(vector):.6f}")
print(f"Max value         : {max(vector):.6f}")

Vector dimensions : 1024
First 10 values   : [-0.10760498046875, 0.003124237060546875, -0.0230712890625, 0.0156097412109375, 0.0286712646484375, 0.0281524658203125, -0.006671905517578125, 0.033935546875, 0.0129241943359375, 0.019683837890625]
Last  10 values   : [0.02471923828125, 0.0186767578125, 0.048736572265625, 0.0093231201171875, -0.033721923828125, -0.048126220703125, 0.006641387939453125, -0.03839111328125, -0.00080108642578125, -0.0079803466796875]
Min value         : -0.154785
Max value         : 0.220459


Embed Multiple Documents

In [6]:
texts = [
    "Competitive analysis helps identify market position.",
    "SWOT analysis is a common framework.",
    "Porter's five forces is another approach.",
]

vectors = embeddings.embed_documents(texts)

print(f"Total docs embedded : {len(vectors)}")
print(f"Each vector size    : {len(vectors[0])}")

for i, (text, vec) in enumerate(zip(texts, vectors)):
    print(f"\n  doc #{i+1} : '{text[:50]}'")
    print(f"  first 5 values : {vec[:5]}")

Total docs embedded : 3
Each vector size    : 1024

  doc #1 : 'Competitive analysis helps identify market positio'
  first 5 values : [-0.08746337890625, -0.007678985595703125, -0.0438232421875, -0.006526947021484375, -0.007617950439453125]

  doc #2 : 'SWOT analysis is a common framework.'
  first 5 values : [-0.061004638671875, 0.0165863037109375, -0.076416015625, -0.017822265625, 0.001003265380859375]

  doc #3 : 'Porter's five forces is another approach.'
  first 5 values : [0.0255584716796875, -0.003520965576171875, -0.054779052734375, 0.029327392578125, 0.00043964385986328125]


In [9]:
for i, (text, vec) in enumerate(zip(texts, vectors)):
    print(f"\n doc #{i+1} :  '{text[:50]}'")
    print(f"first 5 values : {vec[:5]}")


 doc #1 :  'Competitive analysis helps identify market positio'
first 5 values : [-0.08746337890625, -0.007678985595703125, -0.0438232421875, -0.006526947021484375, -0.007617950439453125]

 doc #2 :  'SWOT analysis is a common framework.'
first 5 values : [-0.061004638671875, 0.0165863037109375, -0.076416015625, -0.017822265625, 0.001003265380859375]

 doc #3 :  'Porter's five forces is another approach.'
first 5 values : [0.0255584716796875, -0.003520965576171875, -0.054779052734375, 0.029327392578125, 0.00043964385986328125]


Parse PDF with Unstructured.io

In [10]:
import os
import json
from collections import defaultdict
from unstructured_pytesseract import pytesseract
from unstructured.partition.pdf import partition_pdf

# Windows path — adjust if needed
pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
os.environ["TESSDATA_PREFIX"] = r"C:\Program Files\Tesseract-OCR\tessdata"

os.makedirs("./extracted_images", exist_ok=True)

elements = partition_pdf(
    filename="attention.pdf",
    strategy="hi_res",
    infer_table_structure=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_output_dir="./extracted_images",
    extract_image_block_to_payload=False,
    include_page_breaks=True,
    languages=["eng"],
    detect_language_per_element=True,
    hi_res_model_name="yolox",
    unique_element_ids=True,
    starting_page_number=1,
)

print(f"Total elements extracted: {len(elements)}")

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Total elements extracted: 247


Element Type Distribution

In [11]:
grouped = defaultdict(list)
for el in elements:
    grouped[type(el).__name__].append(el)

print("\nType distribution:")
for type_name, els in sorted(grouped.items()):
    print(f"  {type_name:25s} : {len(els)}")


Type distribution:
  FigureCaption             : 5
  Footer                    : 7
  Formula                   : 4
  Header                    : 3
  Image                     : 7
  ListItem                  : 43
  NarrativeText             : 81
  PageBreak                 : 15
  Table                     : 4
  Text                      : 52
  Title                     : 26


Semantic Chunking (chunk_by_title) → LangChain Docs

In [13]:
from langchain_core.documents import Document
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(elements, max_characters=1000, overlap=50)

langchain_docs = [
    Document(
        page_content=chunk.text,
        metadata={
            "source" : chunk.metadata.filename,
            "page"   : chunk.metadata.page_number,
            "type"   : type(chunk).__name__,
        }
    )
    for chunk in chunks
    if chunk.text.strip()
]

print(f"Total chunks after chunk_by_title : {len(chunks)}")
print(f"Non-empty LangChain docs          : {len(langchain_docs)}")
print(f"\nSample chunk:")
print(f"  page    : {langchain_docs[5].metadata['page']}")
print(f"  type    : {langchain_docs[5].metadata['type']}")
print(f"  content : {langchain_docs[5].page_content[:150]}")

Total chunks after chunk_by_title : 54
Non-empty LangChain docs          : 54

Sample chunk:
  page    : 2
  type    : CompositeElement
  content : Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computa


Create Qdrant Collection (Dense 1024-dim + Sparse BM25)

In [20]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.http.models import (
    VectorParams, Distance, HnswConfigDiff, Datatype,
    SparseVectorParams, SparseIndexParams, Modifier
)

COLLECTION_NAME       = "multimodal-rag"
IMAGE_COLLECTION_NAME = "multimodal-rag-images"
DENSE_DIM             = 1024   # baai/bge-m3 output dimension

# ─── Connect to Qdrant Cloud ───────────────────────────────────────────────
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

# ─── Alternatively, use local persistent storage ───────────────────────────
# client = QdrantClient(path="./qdrant_local")


def create_collection(name: str):
    """Create collection with dense + sparse vectors if it does not exist."""
    existing = [c.name for c in client.get_collections().collections]
    if name in existing:
        print(f"Collection '{name}' already exists — skipping creation.")
        return

    client.create_collection(
        collection_name=name,
        vectors_config={
            "dense": VectorParams(
                size=DENSE_DIM,
                distance=Distance.COSINE,
                on_disk=False,              # keep in RAM for low-latency
                hnsw_config=HnswConfigDiff(
                    m=24,                   # connectivity (higher = better recall, more RAM)
                    ef_construct=256,       # build-time accuracy
                    payload_m=24,           # HNSW indexes payload fields too
                ),
                datatype=Datatype.FLOAT32,
            )
        },
        # ── Sparse vector config (BM25-style IDF, matches your PUT) ──
        sparse_vectors_config={
            "sparse": SparseVectorParams(
                index=SparseIndexParams(on_disk=True),   # inverted index on disk
                modifier=Modifier.IDF,                   # IDF weighting = BM25 behaviour
            )
        },
    )
    print(f"Collection '{name}' created.")


create_collection(COLLECTION_NAME)
create_collection(IMAGE_COLLECTION_NAME)

info = client.get_collection(COLLECTION_NAME)
print(f"Collection status : {info.status}")
print(f"Points count      : {info.points_count}")
print(f"Indexed vectors   : {info.indexed_vectors_count}")
print(f"Segments count    : {info.segments_count}")
print(f"Payload schema    : {info.payload_schema}")

Collection 'multimodal-rag' already exists — skipping creation.
Collection 'multimodal-rag-images' already exists — skipping creation.
Collection status : green
Points count      : 0
Indexed vectors   : 0
Segments count    : 2
Payload schema    : {}


Build QdrantVectorStore (Hybrid Mode) + Upsert Chunks

In [22]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

# Sparse embeddings: FastEmbed BM25 (runs locally, no API key needed)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

# Wrap existing collection with QdrantVectorStore (HYBRID mode)
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,                   # NVIDIA bge-m3 (dense 1024-dim)
    sparse_embedding=sparse_embeddings,     # FastEmbed BM25  (sparse)
    retrieval_mode=RetrievalMode.HYBRID,    # dense + sparse + RRF fusion
    vector_name="dense",
    sparse_vector_name="sparse",
)

# Upsert chunked docs
vectorstore.add_documents(langchain_docs)

# Verify
info = client.get_collection(COLLECTION_NAME)
print(f"Points in Qdrant collection : {info.points_count}")

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Points in Qdrant collection : 54


Inspect Collection Info & Sample Points

In [23]:
# Collection stats
info = client.get_collection(COLLECTION_NAME)
print(f"Collection   : {COLLECTION_NAME}")
print(f"Points       : {info.points_count}")
print(f"Status       : {info.status}")

# Scroll first 3 points
points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=3,
    with_payload=True,
    with_vectors=False,
)

print(f"\nSample stored points:")
for p in points:
    payload = p.payload or {}
    meta    = payload.get("metadata", {})
    content = payload.get("page_content", "")[:120]
    print(f"\n  ID     : {p.id}")
    print(f"  page   : {meta.get('page')}")
    print(f"  type   : {meta.get('type')}")
    print(f"  text   : {content}")

Collection   : multimodal-rag
Points       : 54
Status       : green

Sample stored points:

  ID     : 10d9d73b-ba9f-4219-a814-23e20bf2bce2
  page   : 11
  type   : CompositeElement
  text   : [11] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for im- age recognition. In Proceedin

  ID     : 16abd497-a639-439f-99ea-e0662fc0274e
  page   : 7
  type   : CompositeElement
  text   : A single convolutional layer with kernel width k < n does not connect all pairs of input and output positions. Doing so 

  ID     : 1e106bf6-585e-4a93-863c-a8ab20dc928f
  page   : 6
  type   : CompositeElement
  text   : 4 Why Self-Attention

In this section we compare various aspects of self-attention layers to the recurrent and convolu- 


Dense-Only Similarity Search Test

In [24]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

# Dense-only retriever for comparison
vectorstore_dense = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    retrieval_mode=RetrievalMode.DENSE,
    vector_name="dense",
)


In [25]:
query = "What is self attention"
results = vectorstore_dense.similarity_search_with_score(query, k=3)

print(f"\nTop {len(results)} DENSE results for: '{query}'")
for i, (doc, score) in enumerate(results, start=1):
    print(f"\n  Result #{i} | score={score:.4f}")
    print(f"  page    : {doc.metadata.get('page')}")
    print(f"  type    : {doc.metadata.get('type')}")
    print(f"  content : {doc.page_content[:200]}")


Top 3 DENSE results for: 'What is self attention'

  Result #1 | score=0.6348
  page    : 2
  type    : CompositeElement
  content : Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has be

  Result #2 | score=0.5247
  page    : 5
  type    : CompositeElement
  content : • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward informatio

  Result #3 | score=0.5119
  page    : 3
  type    : CompositeElement
  content : 3.2 Attention

An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as


Hybrid Search Test (Dense + Sparse + RRF Fusion)

In [26]:
# Hybrid vectorstore (already built in Cell 10, reuse it)
query = "What is self attention"
results = vectorstore.similarity_search_with_score(query, k=3)

print(f"\nTop {len(results)} HYBRID results for: '{query}'")
for i, (doc, score) in enumerate(results, start=1):
    print(f"\n  Result #{i} | score={score:.4f}")
    print(f"  page    : {doc.metadata.get('page')}")
    print(f"  type    : {doc.metadata.get('type')}")
    print(f"  content : {doc.page_content[:200]}")


Top 3 HYBRID results for: 'What is self attention'

  Result #1 | score=1.0000
  page    : 2
  type    : CompositeElement
  content : Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has be

  Result #2 | score=0.3333
  page    : 6
  type    : CompositeElement
  content : 4 Why Self-Attention

In this section we compare various aspects of self-attention layers to the recurrent and convolu- tional layers commonly used for mapping one variable-length sequence of symbol r

  Result #3 | score=0.3333
  page    : 5
  type    : CompositeElement
  content : • Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward informatio


Save Elements to JSON (for reproducibility)

In [27]:
from unstructured.staging.base import elements_to_json, elements_from_json

elements_to_json(elements, filename="attention_all_elements.json", indent=2)
print("All elements saved to attention_all_elements.json")

reloaded = elements_from_json(filename="attention_all_elements.json")
print(f"Reloaded {len(reloaded)} elements from JSON")

All elements saved to attention_all_elements.json
Reloaded 247 elements from JSON


Inspect JSON Keys & Metadata Fields

In [28]:
import json

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

top_keys  = sorted({k for record in data for k in record.keys()})
meta_keys = sorted({k for record in data for k in record.get("metadata", {}).keys()})

print(f"Top-level keys : {top_keys}")
print(f"Metadata keys  : {meta_keys}")

Top-level keys : ['element_id', 'metadata', 'text', 'type']
Metadata keys  : ['coordinates', 'detection_class_prob', 'filename', 'filetype', 'image_path', 'is_extracted', 'languages', 'last_modified', 'page_number', 'parent_id', 'text_as_html']


Inspect Images + OCR Garbage Detection

In [29]:
import json
import re

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

images = [r for r in data if r["type"] == "Image"]

GARBAGE_PATTERNS = [
    r"<ped>", r"<SOd>", r"<SOA>",
    r"[£¥§©®°±²³¼½¾]",
    r"\b[A-Z]{6,}\b",
    r"={3,}",
    r"\w{1}\s\w{1}\s\w{1}\s\w{1}",
]

def is_garbage(text: str) -> bool:
    if not text or len(text.strip()) < 10:
        return True
    for pattern in GARBAGE_PATTERNS:
        if re.search(pattern, text):
            return True
    non_ascii = sum(1 for c in text if ord(c) > 127)
    if non_ascii / len(text) > 0.20:
        return True
    return False

print(f"Total images: {len(images)}")
for i, img in enumerate(images, start=1):
    text       = img["text"] or ""
    image_path = img["metadata"].get("image_path") or "not saved"
    status     = "GARBAGE" if is_garbage(text) else "USABLE"

    print(f"\nImage #{i} | page={img['metadata'].get('page_number')} | {status}")
    print(f"  disk path : {image_path}")
    print(f"  chars={len(text)} | words={len(text.split())}")
    print(f"  preview   : {text[:120] if text else 'empty'}")

Total images: 7

Image #1 | page=3 | USABLE
  disk path : ./extracted_images\figure-3-1.jpg
  chars=319 | words=54
  preview   : Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx

Image #2 | page=4 | GARBAGE
  disk path : ./extracted_images\figure-4-2.jpg
  chars=0 | words=0
  preview   : empty

Image #3 | page=4 | USABLE
  disk path : ./extracted_images\figure-4-3.jpg
  chars=36 | words=6
  preview   : EEE Scaled Dot-Product | Attention 4

Image #4 | page=13 | GARBAGE
  disk path : ./extracted_images\figure-13-4.jpg
  chars=384 | words=89
  preview   : n £ c < c 2 2 > & oO n a= Ze i) o > s o 8 =| HPAANAAAAA Ez, 8 Boeegse8Be, 42 PS8F SERRERR 268 PT, FESGaeavoezreosHePi_se

Image #5 | page=14 | GARBAGE
  disk path : ./extracted_images\figure-14-5.jpg
  chars=458 | words=96
  preview   : <ped> <ped> UOIUIGO == = uoluIdo Aw — Aw ul ul Bulssiw Bulssiw ae » ale aM: aM JEUM « yeEUM sl sl SIU] SIU} ysn/ isn aq 

Image #6 |

Build Filtered LangChain Docs (skip garbage OCR + PageBreaks)

In [30]:
import json
import re
from collections import Counter
from langchain_core.documents import Document

with open("attention_all_elements.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ── same garbage filter as original ─────────────────────────────────────────
GARBAGE_PATTERNS = [
    r"<ped>", r"<SOd>", r"<SOA>",
    r"[£¥§©®°±²³¼½¾]",
    r"\b[A-Z]{6,}\b",
    r"={3,}",
    r"\w{1}\s\w{1}\s\w{1}\s\w{1}",
]

def is_garbage(text: str) -> bool:
    if not text or len(text.strip()) < 10:
        return True
    for pattern in GARBAGE_PATTERNS:
        if re.search(pattern, text):
            return True
    non_ascii = sum(1 for c in text if ord(c) > 127)
    if non_ascii / len(text) > 0.20:
        return True
    return False

langchain_docs = []
skipped        = []

for record in data:
    text = record["text"] or ""

    if not text.strip():
        skipped.append((record["type"], "empty text"))
        continue

    if record["type"] == "Image" and is_garbage(text):
        skipped.append(("Image", f"page={record['metadata'].get('page_number')} garbage OCR"))
        continue

    langchain_docs.append(Document(
        page_content=text,
        metadata={
            "type"       : record["type"],
            "page"       : record["metadata"].get("page_number"),
            "element_id" : record["element_id"],
            "image_path" : record["metadata"].get("image_path"),
        }
    ))

type_counts = Counter(d.metadata["type"] for d in langchain_docs)
print(f"Skipped ({len(skipped)}):")
for t, reason in skipped:
    print(f"  [{t}] {reason}")

print(f"\nTotal docs going into Qdrant: {len(langchain_docs)}")
for t, c in sorted(type_counts.items()):
    print(f"  {t:20s} : {c}")

Skipped (20):
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [Image] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [PageBreak] empty text
  [Image] page=13 garbage OCR
  [PageBreak] empty text
  [Image] page=14 garbage OCR
  [PageBreak] empty text
  [Image] page=15 garbage OCR
  [Image] page=15 garbage OCR
  [PageBreak] empty text

Total docs going into Qdrant: 227
  FigureCaption        : 5
  Footer               : 7
  Formula              : 4
  Header               : 3
  Image                : 2
  ListItem             : 43
  NarrativeText        : 81
  Table                : 4
  Title                : 26
  UncategorizedText    : 52


Rebuild Qdrant Collection with Clean Docs

This deletes + recreates the collection so you start fresh with filtered data.

In [31]:
from qdrant_client.http.models import (
    VectorParams, Distance, HnswConfigDiff, Datatype,
    SparseVectorParams, SparseIndexParams, Modifier
)
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

# ── Delete old collection and recreate with clean docs ──────────────────────
client.delete_collection(COLLECTION_NAME)
print(f"Deleted old collection: {COLLECTION_NAME}")

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(
            size=1024,
            distance=Distance.COSINE,
            on_disk=False,
            hnsw_config=HnswConfigDiff(m=24, ef_construct=256, payload_m=24),
            datatype=Datatype.FLOAT32,
        )
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=True),
            modifier=Modifier.IDF,
        )
    },
)
print(f"Recreated collection: {COLLECTION_NAME}")

# ── Sparse embedder ──────────────────────────────────────────────────────────
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

# ── Hybrid vectorstore ───────────────────────────────────────────────────────
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

# ── Upsert all clean docs ─────────────────────────────────────────────────────
vectorstore.add_documents(langchain_docs)

# ── Verify ─────────────────────────────────────────────────────────────────────
info = client.get_collection(COLLECTION_NAME)
print(f"\nQdrant index size : {info.points_count}")

Deleted old collection: multimodal-rag
Recreated collection: multimodal-rag

Qdrant index size : 227


Multi-Query Similarity Search with Scores

In [33]:
from collections import Counter

queries = [
    # "What is the transformer architecture?",
    "explain the positional encoding",
    # "What is Multi-Head Attention",
]

for q in queries:
    print(f"QUERY: {q}")

    results = vectorstore.similarity_search_with_score(q, k=4)

    type_hits = Counter(doc.metadata.get("type") for doc, _ in results)
    print(f"Type hits in top 4: {dict(type_hits)}")

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\n  Rank #{rank} | score={score:.4f} | type={doc.metadata.get('type')} | page={doc.metadata.get('page')}")
        print(f"  content : {doc.page_content[:200]}")

QUERY: explain the positional encoding
Type hits in top 4: {'NarrativeText': 3, 'Title': 1}

  Rank #1 | score=0.7500 | type=NarrativeText | page=6
  content : Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of t

  Rank #2 | score=0.5000 | type=Title | page=6
  content : 3.5 Positional Encoding

  Rank #3 | score=0.3333 | type=NarrativeText | page=3
  content : Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head at

  Rank #4 | score=0.3333 | type=NarrativeText | page=6
  content : where pos is the position and i is the dimension. That is, each dimension of the positional encoding corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We c


Build Separate Image Vectorstore (Qdrant Collection)

In [34]:
from qdrant_client.http.models import (
    VectorParams, Distance, HnswConfigDiff, Datatype,
    SparseVectorParams, SparseIndexParams, Modifier
)
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

# ── Separate image-only docs ─────────────────────────────────────────────────
image_docs_only = [
    doc for doc in langchain_docs
    if doc.metadata.get("type") == "Image"
]
print(f"Image docs for separate store: {len(image_docs_only)}")
for doc in image_docs_only:
    print(f"  page={doc.metadata['page']} | {doc.page_content[:80]}")

# ── Recreate image collection ─────────────────────────────────────────────────
if IMAGE_COLLECTION_NAME in [c.name for c in client.get_collections().collections]:
    client.delete_collection(IMAGE_COLLECTION_NAME)

client.create_collection(
    collection_name=IMAGE_COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(
            size=1024,
            distance=Distance.COSINE,
            on_disk=False,
            hnsw_config=HnswConfigDiff(m=24, ef_construct=256, payload_m=24),
            datatype=Datatype.FLOAT32,
        )
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=True),
            modifier=Modifier.IDF,
        )
    },
)

# ── Image-only vectorstore ────────────────────────────────────────────────────
image_vectorstore = QdrantVectorStore(
    client=client,
    collection_name=IMAGE_COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

if image_docs_only:
    image_vectorstore.add_documents(image_docs_only)

print(f"\nImage collection points: {client.get_collection(IMAGE_COLLECTION_NAME).points_count}")

Image docs for separate store: 2
  page=3 | Output Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, 
  page=4 | EEE Scaled Dot-Product | Attention 4

Image collection points: 2


Load Models for RAG Pipeline

In [35]:
import base64
from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv()

from qdrant_client import QdrantClient
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage

QDRANT_URL          = os.getenv("QDRANT_CLUSTER_URL", "")
QDRANT_API_KEY      = os.getenv("QDRANT_API_KEY", "")
COLLECTION_NAME     = "multimodal-rag"
IMG_COLLECTION_NAME = "multimodal-rag-images"

# ── Clients & Models ────────────────────────────────────────────────────────
client           = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings       = NVIDIAEmbeddings(model="baai/bge-m3")
sparse_embeddings= FastEmbedSparse(model_name="Qdrant/bm25")
text_llm         = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")
vision_llm       = ChatNVIDIA(model="meta/llama-3.2-11b-vision-instruct")

# ── Reconnect to persistent Qdrant collection (no load_local needed) ────────
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

image_vectorstore = QdrantVectorStore(
    client=client,
    collection_name=IMG_COLLECTION_NAME,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse",
)

# ── Retrievers ───────────────────────────────────────────────────────────────
# Main text retriever (k=4 by default)
text_retriever  = vectorstore.as_retriever(search_kwargs={"k": 4})

# Image retriever — pull all stored images
img_count       = client.get_collection(IMG_COLLECTION_NAME).points_count or 10
image_retriever = image_vectorstore.as_retriever(search_kwargs={"k": img_count})

print(f"Vectorstore points (text+all) : {client.get_collection(COLLECTION_NAME).points_count}")
print(f"Image store points            : {img_count}")
print("Models loaded successfully.")

Vectorstore points (text+all) : 227
Image store points            : 2
Models loaded successfully.


retrieve_with_images Function (Qdrant Hybrid Retrieval)

In [36]:
def retrieve_with_images(query: str, k: int = 4) -> tuple[list, list]:
    """
    Retrieves text docs via hybrid search (dense+sparse RRF)
    and all image docs from the separate image collection.
    """
    results    = vectorstore.similarity_search_with_score(query, k=k)
    text_docs  = []
    image_docs = []

    for doc, score in results:
        doc.metadata["score"] = round(score, 4)
        if doc.metadata.get("type") == "Image":
            image_docs.append(doc)
        else:
            text_docs.append(doc)

    # Always pull all images from dedicated image collection
    extra_image_docs = image_retriever.invoke(query)
    for doc in extra_image_docs:
        if doc not in image_docs:
            image_docs.append(doc)

    return text_docs, image_docs


def print_retrieved(text_docs: list, image_docs: list) -> None:
    print(f"\nText docs retrieved  : {len(text_docs)}")
    for i, doc in enumerate(text_docs, start=1):
        print(f"\n  [{i}] type={doc.metadata.get('type')} | page={doc.metadata.get('page')} | score={doc.metadata.get('score', 'N/A')}")
        print(f"       {doc.page_content[:200]}")

    print(f"\nImage docs retrieved : {len(image_docs)}")
    for i, doc in enumerate(image_docs, start=1):
        print(f"\n  [{i}] page={doc.metadata.get('page')} | score={doc.metadata.get('score', 'N/A')}")
        print(f"       path={doc.metadata.get('image_path')}")
        print(f"       ocr ={doc.page_content[:100]}")

print("retrieve_with_images defined.")

retrieve_with_images defined.


nswer_from_text Function (Text LLM RAG)

In [37]:
def answer_from_text(query: str, text_docs: list) -> str:
    context = "\n\n".join(
        f"[Page {d.metadata.get('page')} | {d.metadata.get('type')}]\n{d.page_content}"
        for d in text_docs
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Answer the question based only on the context below.\n"
         "If the answer is not in the context, say 'Not found in document'.\n\n"
         "Context:\n{context}"),
        ("human", "{question}"),
    ])

    chain = prompt | text_llm | StrOutputParser()
    return chain.invoke({"context": context, "question": query})

print("answer_from_text defined.")

answer_from_text defined.


answer_from_images Function (Vision LLM)

In [38]:
def answer_from_images(query: str, image_docs: list) -> str:
    if not image_docs:
        return "No relevant images found."

    answers = []
    for doc in image_docs:
        image_path = doc.metadata.get("image_path")
        if not image_path or not Path(image_path).exists():
            continue

        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")

        ext  = Path(image_path).suffix.lower().replace(".", "")
        mime = f"image/{'jpeg' if ext in ['jpg', 'jpeg'] else ext}"

        response = vision_llm.invoke([
            HumanMessage(content=[
                {"type": "text",
                 "text": f"This image is from page {doc.metadata.get('page')} of a research paper. {query}"},
                {"type": "image_url",
                 "image_url": {"url": f"data:{mime};base64,{b64}"}},
            ])
        ])
        answers.append(f"[Image page {doc.metadata.get('page')}]: {response.content}")

    return "\n\n".join(answers) if answers else "No valid image paths found."

print("answer_from_images defined.")

answer_from_images defined.


multimodal_chat Function (Full Pipeline)

In [39]:
def multimodal_chat(query: str, k: int = 4) -> None:
    print(f"QUERY: {query}")

    # Step 1 — Hybrid retrieval from Qdrant
    text_docs, image_docs = retrieve_with_images(query, k=k)
    print_retrieved(text_docs, image_docs)

    # Step 2 — Text LLM answers from text context
    if text_docs:
        text_answer = answer_from_text(query, text_docs)
        print(text_answer)

    # Step 3 — Vision LLM answers from image context
    if image_docs:
        image_answer = answer_from_images(query, image_docs)
        if image_answer not in ("No relevant images found.", "No valid image paths found."):
            print(f"\n--- Image Analysis ---")
            print(image_answer)

print("multimodal_chat defined.")

multimodal_chat defined.


Query: Transformer Architecture

In [40]:
multimodal_chat("What is the transformer architecture?", k=4)

QUERY: What is the transformer architecture?

Text docs retrieved  : 4

  [1] type=FigureCaption | page=3 | score=1.0
       Figure 1: The Transformer - model architecture.

  [2] type=NarrativeText | page=2 | score=0.6667
       In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Tran

  [3] type=NarrativeText | page=3 | score=0.45
       The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, 

  [4] type=NarrativeText | page=10 | score=0.25
       In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with m

Image docs retrieved : 2

  [1] page=4 | score=N/A
    

Query: Positional Encoding

In [41]:
multimodal_chat("explain the positional encoding", k=4)

QUERY: explain the positional encoding

Text docs retrieved  : 4

  [1] type=NarrativeText | page=6 | score=0.75
       Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of t

  [2] type=Title | page=6 | score=0.5
       3.5 Positional Encoding

  [3] type=NarrativeText | page=6 | score=0.3333
       where pos is the position and i is the dimension. That is, each dimension of the positional encoding corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We c

  [4] type=NarrativeText | page=3 | score=0.3333
       Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head at

Image docs retrieved : 2

  [1] page=3 | score=N/A
       path=./extracted_images\figure-3-1

In [ ]:
multimodal_chat("What is Multi-Head Attention", k=4)

In [42]:
multimodal_chat("What does the model architecture diagram show?", k=4)

QUERY: What does the model architecture diagram show?

Text docs retrieved  : 4

  [1] type=Title | page=2 | score=0.75
       3 Model Architecture

  [2] type=FigureCaption | page=3 | score=0.5333
       Figure 1: The Transformer - model architecture.

  [3] type=NarrativeText | page=1 | score=0.5
       The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and

  [4] type=NarrativeText | page=10 | score=0.3333
       Our results in Table 4 show that despite the lack of task-specific tuning our model performs sur- prisingly well, yielding better results than all previously reported models with the exception of the 

Image docs retrieved : 2

  [1] page=4 | score=N/A
       path=./extracted_images\figure-4-3.jpg
       ocr =EEE Scaled Dot-Product | Attention 4

  [2] page=3 | score=N/A
       path=./extracted_images\figure-3-1.jpg
       ocr =Ou

Exception: [504] Gateway Timeout
{'_content': b'', '_content_consumed': True, '_next': None, 'status_code': 504, 'headers': {'Date': 'Wed, 25 Mar 2026 05:30:28 GMT', 'Content-Length': '0', 'Connection': 'keep-alive', 'Access-Control-Expose-Headers': 'nvcf-reqid', 'Nvcf-Reqid': 'ba868661-b7ff-4038-9112-c647a5094042', 'Nvcf-Status': 'errored', 'Vary': 'Origin'}, 'raw': <urllib3.response.HTTPResponse object at 0x0000029685B60E20>, 'url': 'https://ai.api.nvidia.com/v1/gr/meta/llama-3.2-11b-vision-instruct/chat/completions', 'encoding': None, 'history': [], 'reason': 'Gateway Timeout', 'cookies': <RequestsCookieJar[]>, 'elapsed': datetime.timedelta(seconds=302, microseconds=431680), 'request': <PreparedRequest [POST]>, 'connection': <requests.adapters.HTTPAdapter object at 0x000002968E9FB850>}

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

all_models = ChatNVIDIA.get_available_models()
vision_models = [
    m for m in all_models
    if any(k in m.id.lower() for k in ["neva", "vision", "vlm", "visual", "llava", "llama-3.2"])
]

print("Available vision models:")
for m in vision_models:
    print(f"  {m.id}")

```js
attention.pdf
    └─► unstructured partition_pdf  (hi_res, yolox, OCR)
            └─► 247 elements (Title, NarrativeText, Image, Table, ...)
                    └─► chunk_by_title (max 1000 chars, overlap 50)
                            └─► is_garbage() filter  (skip PageBreaks + OCR garbage)
                                    └─► 227 LangChain Documents
                                            ├─► NVIDIA baai/bge-m3  (dense 1024-dim)
                                            ├─► FastEmbed BM25       (sparse IDF)
                                            └─► QdrantVectorStore    (HYBRID mode)
                                                    ├─► collection: multimodal-rag        (all docs)
                                                    └─► collection: multimodal-rag-images (images only)

Query  →  Hybrid Search (dense prefetch + sparse prefetch + RRF fusion)
        ├─► text_docs  →  ChatNVIDIA llama-3.1-8b  →  text answer
        └─► image_docs →  ChatNVIDIA llama-3.2-vision  →  image answer
```

| Component | Original (FAISS) | Replacement (Qdrant) |
|---|---|---|
| Vector index | `FAISS.from_documents(...)` | `QdrantVectorStore(... HYBRID)` |
| Dense embed | NVIDIA bge-m3 1024d | Same |
| Sparse embed |  none | FastEmbed BM25 (IDF) |
| Retrieval | `similarity_search_with_score` | `similarity_search_with_score` (same API) |
| Persistence | `save_local` / `load_local` |  Always-on (cloud or local path) |
| Image store | Separate FAISS | Separate Qdrant collection |
| HNSW | FAISS default | m=24, ef_construct=256, payload_m=24 |